# segSHAPE Reproduction Report

Reproduction of the benchmark results reported in the segSHAPE manuscript using the
released `segshape` package.

---

## v0.1.1 — 2026-06-21

**Manuscript:** https://www.biorxiv.org/content/10.64898/2026.06.15.732177v1

Reproduces the manuscript benchmarks with the installed `segshape` **0.1.1** package.

### 1. Hardware / environment

| component | version |
|---|---|
| segSHAPE | 0.1.1 (`pip install segshape==0.1.1`) |
| ViennaRNA (RNAfold) | 2.7.2 |
| Python | 3.10 |
| numpy / numba / scipy | 2.2.6 / 0.65.1 / 1.15.3 |
| scikit-learn / pyarrow / pod5 | 1.7.2 / 18.0.0 / 0.3.23 |
| pandas / pysam | 2.3.3 / 0.24.0 |
| CPU | AMD EPYC 7003-series (Milan, Zen 3) |

> **Reproducibility note.** The reference numbers below were generated on AMD EPYC 7003-series
> (Milan, Zen 3) CPUs. Segmentation uses floating-point comparisons that can differ marginally between CPU
> architectures, which can shift a small number of sub-events; the structure-level metrics
> are robust, but for bit-identical intermediate files use the same CPU family.

### 2. Parameters

Unified parameter set; only the chemistry-dependent options differ across datasets.
Pipeline: `dorado-extract → segment → event-align (control→treated) → mod-calling (treated)
→ RNAfold → evaluate`.

```
segment       --norm med-mad --trim 0.1 --resplit-std 0.15
              --resplit-max-pieces 2 --resplit-min-piece-len 4
event-align   --delta-event 50 --delta-kmer 15 --boundary-cost 0.0 --skip-penalty 50
              --sigma-multiplier 1.5 --kappa 50 --length-weight capped --length-weight-cap 30
              --fit-mode shift_only --shift-bounds ±1 (RNA002) / ±0.3 (RNA004)
mod-calling   --method if-1D --smooth-window 0 --normalize zscore
RNAfold       -p -d2 --noLP --noPS -P andronescu2007.par --shapeMethod=D
```

Per-dataset configuration:

| dataset | chemistry | contig | dorado-extract filter | shift-bounds | contamination |
|---|---|---|---|---|---|
| miR17-92 | RNA002 | miR17-92 | — | ±1 | 0.005 |
| tetra | RNA002 | tetra | — | ±1 | 0.005 |
| bsub_16S | RNA002 | bacillus_subtilis_16S | `--min-ref-coverage 0.5` | ±1 | 0.005 |
| smPC_002pool_wt | RNA002 | TETRA | `--keep-contig TETRA` | ±1 | 0.005 |
| smPC_004pool_wt | RNA004 | TETRA | `--keep-contig TETRA` | ±0.3 | 0.02 |

### 3. Results

Centroid-structure **precision / recall / F1 / MCC** vs ground truth, computed by the cells
below directly from the shipped `rnafold_<method>.out` (centroid structure) and
`structure_gt.txt` — pure Python, no `segshape` install needed. Run from this directory
(`release/segSHAPE/datasets/`).

`if-1D` is the manuscript's mod-calling method — its **MCC** reproduces the manuscript
(87.04 / 76.73 / 57.53 / 73.45 / 72.56). `gmm-1D` is an alternative, fully-deterministic
method (quantile q = 0.005 RNA002 / 0.02 RNA004); GMM is **not** reported in this version of
the manuscript for smPC_002pool_wt / smPC_004pool_wt.

In [1]:
import pandas as pd

def read_dotbracket(path):
    """Concatenate the dot-bracket line(s) from a structure file, dropping > / # headers."""
    parts = []
    for line in open(path):
        if line.startswith((">", "#")):
            continue
        parts.append(line.strip())
    return "".join(parts)

def parse_centroid(rnafold_out):
    """Centroid structure from an RNAfold `-p` output.
    RNAfold prints structures in the order MFE, ensemble(prob), centroid. Keeping only
    pure '.()' lines drops the ensemble line (it contains '{ } | ,'), leaving
    [MFE, centroid] -- the centroid is the 2nd. (The file's last line is the
    frequency/diversity text, not a structure.)"""
    db = []
    for line in open(rnafold_out):
        tok = line.strip().split(" ")[0]
        if len(tok) >= 10 and set(tok) <= set(".()"):
            db.append(tok)
    return db[1] if len(db) >= 2 else db[0]

def find_pairs(s):
    """Stack-based base-pair extraction -> set of 1-indexed (i, j)."""
    stack, pairs = [], []
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        elif ch == ")" and stack:
            pairs.append((stack.pop() + 1, i + 1))
    return set(pairs)

def struct_metrics(y_true, y_pred):
    """precision / recall / F1 / MCC over base pairs.
    Negatives = all C(n,2) position pairs that are not base-paired (segSHAPE convention)."""
    assert len(y_true) == len(y_pred), f"length {len(y_true)} != {len(y_pred)}"
    n = len(y_true)
    comb = n * (n - 1) // 2
    yt, yp = find_pairs(y_true), find_pairs(y_pred)
    tp = len(yt & yp); fn = len(yt - yp); fp = len(yp - yt); tn = comb - tp - fn - fp
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    den = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    M = (tp * tn - fp * fn) / den if den else 0.0
    return dict(precision=P*100, recall=R*100, f1=F*100, mcc=M*100, tp=tp, fn=fn, fp=fp)


# (folder, contig-file prefix, chemistry)
DATASETS = [
    ("miR17-92",        "",       "RNA002"),
    ("tetra",           "",       "RNA002"),
    ("bsub_16S",        "",       "RNA002"),
    ("smPC_002pool_wt", "TETRA_", "RNA002"),
    ("smPC_004pool_wt", "TETRA_", "RNA004"),
]

rows = []
for folder, pfx, chem in DATASETS:
    gt = read_dotbracket(f"{folder}/{pfx}structure_gt.txt")
    for method in ("if", "gmm"):
        centroid = parse_centroid(f"{folder}/{pfx}rnafold_{method}.out")
        m = struct_metrics(gt, centroid)
        rows.append(dict(dataset=folder, chemistry=chem, method=f"{method}-1D",
                         precision=round(m["precision"], 2), recall=round(m["recall"], 2),
                         F1=round(m["f1"], 2), MCC=round(m["mcc"], 2)))

df = pd.DataFrame(rows).set_index(["dataset", "method"])
print(df.to_string())

                       chemistry  precision  recall     F1    MCC
dataset         method                                           
miR17-92        if-1D     RNA002      89.84   84.35  87.01  87.04
                gmm-1D    RNA002      85.71   82.44  84.05  84.05
tetra           if-1D     RNA002      83.02   70.97  76.52  76.73
                gmm-1D    RNA002      83.50   69.35  75.77  76.07
bsub_16S        if-1D     RNA002      65.12   50.85  57.11  57.53
                gmm-1D    RNA002      70.44   54.26  61.30  61.81
smPC_002pool_wt if-1D     RNA002      82.65   65.32  72.97  73.45
                gmm-1D    RNA002      79.21   64.52  71.11  71.45
smPC_004pool_wt if-1D     RNA004      93.33   56.45  70.35  72.56
                gmm-1D    RNA004      93.59   58.87  72.28  74.20
